# Tools in LLMs: Solving the Knowledge Cutoff Problem

Welcome to this notebook! By the end you will understand:

1. What LLMs can and cannot do on their own
2. Why the **knowledge cutoff** is a fundamental limitation
3. What **Tools** are and how they fix that limitation
4. The exact step-by-step flow of how tool calling works
5. When tools are the right solution, and when they are **not**

## Section 1: What Can an LLM Do?

Large Language Models (LLMs) like GPT-4, Claude, or Qwen are trained on massive amounts of text. They are genuinely powerful at many tasks:

**What LLMs are great at:**

- **Text Generation**: Writing essays, emails, code, summaries, explanations
- **Reasoning**: Solving multi-step problems, logic puzzles, math
- **Image Understanding**: Describing and analyzing images (multimodal models)
- **Document Understanding**: Reading PDFs, extracting key information
- **Code Writing**: Writing, debugging, and explaining code in many languages
- **Language Translation**: Translating between dozens of languages
- **Conversation**: Answering questions, holding context across a dialogue

> Think of an LLM as an extremely well-read expert who has studied millions of books, articles, and websites — but was locked in a library and stopped receiving new books on a fixed date.

## Section 2: The Big Problem: Knowledge Cutoff

Every LLM is trained on data collected **up to a specific date**. After that date: the model knows nothing about what happened in the world.

This date is called the **Knowledge Cutoff**.

| Model | Approximate Cutoff |
|-------|-------------------|
| GPT-4 | Early 2024 |
| Claude Opus / Sonnet | Early 2025 |
| Qwen 3.5 | Mid 2025 |

### Why is this a problem?

The real world keeps changing: but the model is frozen. This means the model **cannot** answer questions like:

- *"What is the temperature in Pune right now?"*
- *"Who won yesterday's IPL match?"*
- *"What is the latest stock price of Reliance Industries?"*
- *"What happened in the news today?"*

If you ask an LLM these questions without any tools, one of two things happens:

1. The model honestly says: *"I don't have access to real-time information."*
2. The model **hallucinates**: it makes up a confident-sounding but completely wrong answer.

Both are bad outcomes for users who need accurate, up-to-date information.

## Section 3: What Tools Can Solve vs. What They Cannot

Before we build anything, it is important to understand the boundaries. Tools are powerful: but they are not magic.

### What Tools CAN solve

| Problem | Tool Solution |
|---------|--------------|
| Real-time weather | Call a weather API |
| Latest news headlines | Call a news API |
| Current stock prices | Call a finance API |
| Live cricket / sports scores | Call a sports API |
| Currency exchange rates | Call a forex API |
| Database lookups | Query a database |
| Sending emails or messages | Call an email/messaging API |
| Reading files from a system | File system tool |
| Running calculations | Calculator tool |

### What Tools CANNOT solve

Tools are just Python functions: they can only do what you program them to do. They do not give the LLM:

- **Common sense or judgment**: If your tool returns bad data, the LLM cannot magically know it is wrong
- **Reasoning ability**: A tool can fetch data, but the LLM still needs to interpret it correctly
- **Access to private systems**: Tools can only reach APIs and systems you explicitly connect them to
- **Perfect reliability**: APIs go down, rate-limit you, or return errors; your code must handle this

### When you need something OTHER than tools

| Situation | Better Solution |
|-----------|----------------|
| You need the LLM to remember things across many sessions | A **database or memory layer** (e.g., vector database) |
| You need the LLM to reason deeply over a long document | **RAG (Retrieval Augmented Generation)** |
| You need the LLM to plan and take a series of actions | **Agents** (multi-step tool use with planning) |
| You need the LLM to learn new behaviour | **Fine-tuning** the model |
| You need to run code the LLM writes | **Code execution tool** (sandboxed) |

> **Key insight:** Tools solve the *real-time data* problem. They do not solve reasoning, memory, or learning problems. Use the right solution for the right problem.

## Section 4: The Solution: Tools

A **Tool** is simply a Python function that:

1. Receives an input (e.g., a city name, a topic, a stock ticker)
2. Fetches real-world data from an API or system
3. Returns a result as a string
4. Gets passed back to the LLM so it can answer the user

### The Tool Calling Flow — Step by Step

```
User asks a question
        |
        v
LLM receives the question + list of available tools
        |
        v
LLM decides: "Can I answer this from my training data?"
        |
   YES  |  NO
        |
        v (NO path)
LLM selects the right tool and specifies what arguments to pass
        |
        v
Your Python code runs the tool function → gets real data from an API
        |
        v
The tool result is added to the conversation
        |
        v
LLM reads the tool result and writes a natural language answer
        |
        v
User receives an accurate, up-to-date answer
```

Notice that the **LLM never directly calls the API**. It only *requests* that your code call it. Your Python code is the bridge between the LLM and the outside world. This is an important safety boundary.

## Section 5: The Model We Use: Qwen 3.5 via Ollama

**Ollama** is a tool that lets you run open-source LLMs locally on your own computer, no internet required, no API key, completely private.

**Qwen 3.5** is a family of open-source models from Alibaba that supports tools, vision, and reasoning.

| Capability | Supported |
|-----------|----------|
| Vision (Images) | Yes |
| Tool / Function Calling | Yes |
| Thinking / Reasoning Mode | Yes |
| Runs locally via Ollama | Yes |
| Sizes available | 0.8b, 2b, 4b, 9b, 27b, 35b, 122b |

Today we use `qwen3.5:0.8b`, the smallest version. It is fast, runs on any laptop, and is good enough to demonstrate tool calling clearly.

> **Why start small?** Learning tool calling is about understanding the *flow*, not the model size. Once you understand how it works with a small model, scaling up to a larger model is just changing one line of code.

## Step 1: Install Required Libraries

We need two libraries:

- `requests`: to make HTTP calls to external APIs (news, weather, etc.)
- `ollama`: to talk to locally running Ollama models

In [2]:
# Run this once to install the required packages
!pip install requests ollama

'pip' is not recognized as an internal or external command,
operable program or batch file.


## Step 2: Import Libraries

We import the libraries at the top of the notebook. This is standard Python practice — import once, use everywhere.

In [41]:
import requests
import ollama
import json

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Step 3: Define the Tool (Python Function)

This is the most important concept to understand: **a Tool is just a normal Python function**.

There is no magic here. We write a function that:
1. Takes an argument (the topic the user is asking about)
2. Calls a real API (NewsAPI in this case)
3. Parses the response
4. Returns a clean string with the result

The LLM will never see the raw API response, it only sees what our function returns.

In [46]:
def get_news(topic: str) -> str:
    """
    Tool: Fetches the latest news headline for a given topic.

    Parameters:
        topic (str): The subject to search for, e.g. 'IPL Cricket'

    Returns:
        str: A formatted string with the top headline and source
    """
    api_key = "15bc9e9a1b5e4ded8e24960b2d4e5ac4"   # Your NewsAPI key
    url = f"https://newsapi.org/v2/everything?q={topic}&apiKey={api_key}&pageSize=1"

    response = requests.get(url).json()
    articles = response.get("articles", [])

    if not articles:
        return f"No news found for '{topic}'."

    headline = articles[0]["title"]
    source   = articles[0]["source"]["name"]

    return f"Top headline on '{topic}': {headline} (Source: {source})"


print("Tool defined successfully.")

Tool defined successfully.


## Step 4, Test the Tool Directly (Without the LLM)

Before hooking the tool into the LLM, always test it by calling it directly as a Python function.

This is good practice because:
- You can confirm the API is working
- You can see exactly what the LLM will receive as tool output
- If something is wrong, it is much easier to debug here than inside the full LLM flow

In [48]:
# Choose the topic you want news about
topic = "CSK 2026"

# Call the tool directly — no LLM involved yet
news_result = get_news(topic)

print("Tool Output (what the LLM will receive):")
print(news_result)

Tool Output (what the LLM will receive):
Top headline on 'CSK 2026': MS Dhoni ruled out for first two weeks of IPL 2026 with injury, confirm CSK (Source: The Indian Express)


## Step 5, Describe the Tool to the LLM

This is a critical step that many beginners overlook.

The LLM cannot see your Python code. It cannot read your function definition. **You must tell the LLM about the tool using a structured JSON description.**

This description answers three questions for the LLM:
1. **What is the tool called?** (the `name` field, must match your function name exactly)
2. **What does it do?** (the `description` field, write this clearly, the LLM uses it to decide when to call the tool)
3. **What input does it need?** (the `parameters` field, names, types, and descriptions of each argument)

> The quality of your tool description directly affects how well the LLM uses the tool. A vague description = the LLM will call it at the wrong time or pass the wrong arguments.

In [6]:
# This is the tool description in Ollama's expected format
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_news",         # Must match the Python function name exactly
            "description": "Fetches the latest news headline for a given topic from the internet. Use this when the user asks about current events, recent news, or anything that may have happened recently.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The topic to search news for. Be specific. Examples: 'IPL Cricket 2025', 'Artificial Intelligence', 'Reliance Industries stock'"
                    }
                },
                "required": ["topic"]
            }
        }
    }
]

print("Tool description ready.")

Tool description ready.


## Step 6: The Full Tool Calling Flow

Now we bring everything together. Read through this carefully — understanding this flow is the heart of this notebook.

### What happens in each part:

**Part A**: We send the user's question to the LLM along with the list of tools. The LLM decides whether it needs a tool.

**Part B**: If the LLM wants a tool, it tells us: "Call this function with these arguments." We do exactly that.

**Part C**: We send the tool result back to the LLM. The LLM writes a final, natural language answer using that result.

In [7]:
# --- Map of available tools ---
# This dictionary connects the tool name (a string the LLM uses) to the actual Python function
available_tools = {
    "get_news": get_news
}

### Conversation Roles in LLM Tool Use

Every entry in the `messages` list has a **role** that defines *who* or *what* produced that message:

- **`user`** → The human’s input or question  
  *Example:* `"What’s the latest IPL news?"`

- **`assistant`** → The LLM’s reply  
  - Can be plain text (direct answer)  
  - Or a tool plan (e.g., `"I want to call get_news with topic='IPL Cricket'"`)

- **`tool`** → The actual output returned by a Python function or external API  
  *Example:* `"Latest headline: Dhoni leads CSK to victory"`

---

### Why This Matters
- `user` → captures what the human asked.  
- `assistant` → captures what the model decided (answer or tool call).  
- `tool` → captures the *real data* fetched by your code.  

Together, these roles form the **conversation history** that the LLM reads on the next turn to generate a final, natural‑language answer.


In [53]:
# --- User's question ---
user_message = "CSK team 2026"

# --- Start conversation history ---
messages = [
    {"role": "user", "content": user_message}
]

print(f"User: {user_message}")

User: CSK team 2026


### Part A : Send to LLM (First Call)

We send the user's message AND the tool descriptions to the LLM. The LLM responds with either:
- A direct answer (if it can answer from its training data)
- A tool call request (if it needs real-world data)

In [54]:
# --- Part A: Send message to LLM with tool description ---
response = ollama.chat(
    model="qwen3.5:0.8b",
    messages=messages,
    tools=tools          # <-- This is how we give the LLM access to tools
)

assistant_message = response["message"]

print("LLM's raw response (before we process it):")
print(assistant_message)

LLM's raw response (before we process it):
role='assistant' content='' thinking='The user is asking about CSK team in 2026, which is referring to the Chennai Super Kings cricket team. However, the available functions I have access to (get_news) are designed to fetch news headlines for specific topics, not for sports teams.\n\nLet me check what information I can access:\n1. get_news - Fetches news headlines for cricket, AI, and stock-related topics\n2. The function expects a specific topic (like cricket, AI, etc.)\n3. For CSK team 2026, there\'s no news function available that would help me find actual team news or player information for the future.\n\nHowever, since this is a cricket team topic and I can\'t find specific team 2026 news through the available tools, I should use the get_news function with a specific topic that\'s more relevant. Since CSK is a cricket team, I could potentially use "CSK" as the topic to see if it gives me any cricket-related news, though the user\'s questi

### Part B: Execute the Tool

We check if the LLM requested a tool call. If yes, we run the actual Python function and collect the result.

In [55]:
# --- Part B: Check if LLM wants to use a tool ---
if assistant_message.get("tool_calls"):
    print("LLM decided to use a tool.")
    print("-" * 60)

    # Add the assistant's decision to the conversation history
    # This is important, the LLM needs to see its own previous messages
    messages.append(assistant_message)

    # --- Execute each tool the LLM requested ---
    for tool_call in assistant_message["tool_calls"]:

        tool_name = tool_call["function"]["name"]
        tool_args = tool_call["function"]["arguments"]

        print(f"Tool Requested : {tool_name}")
        print(f"Arguments      : {tool_args}")

        # --- Run the actual Python function ---
        tool_function = available_tools[tool_name]
        tool_result   = tool_function(**tool_args)   # e.g. get_news(topic="IPL Cricket")

        print(f"Tool Result    : {tool_result}")
        print("-" * 60)

        # --- Add tool result to conversation history ---
        messages.append({
            "role": "tool",
            "content": tool_result
        })

else:
    print("LLM answered directly without using any tool.")
    print("(This may mean the LLM thought it already knew the answer — or the model is too small to trigger tool use reliably.)")

LLM decided to use a tool.
------------------------------------------------------------
Tool Requested : get_news
Arguments      : {'topic': 'CSK'}
Tool Result    : Top headline on 'CSK': MS Dhoni ruled out for first two weeks of IPL 2026 with injury, confirm CSK (Source: The Indian Express)
------------------------------------------------------------


In [56]:
messages

[{'role': 'user', 'content': 'CSK team 2026'},
 Message(role='assistant', content='', thinking='The user is asking about CSK team in 2026, which is referring to the Chennai Super Kings cricket team. However, the available functions I have access to (get_news) are designed to fetch news headlines for specific topics, not for sports teams.\n\nLet me check what information I can access:\n1. get_news - Fetches news headlines for cricket, AI, and stock-related topics\n2. The function expects a specific topic (like cricket, AI, etc.)\n3. For CSK team 2026, there\'s no news function available that would help me find actual team news or player information for the future.\n\nHowever, since this is a cricket team topic and I can\'t find specific team 2026 news through the available tools, I should use the get_news function with a specific topic that\'s more relevant. Since CSK is a cricket team, I could potentially use "CSK" as the topic to see if it gives me any cricket-related news, though the

### Part C: Final Answer (Second LLM Call)

We send the full conversation, including the tool result, back to the LLM. It now writes a clean, natural language answer for the user.

In [59]:
# --- Part C: Send the tool result back to LLM for final answer ---
final_response = ollama.chat(
    model="qwen3.5:0.8b",
    messages=messages        # Now includes: user message + tool call + tool result
)

final_answer = final_response["message"]["content"]

print("Final Answer from LLM:")
print(final_answer)

Final Answer from LLM:
# CSK Team 2026 Updates

## Latest Developments
MS Dhoni has been ruled out of first two weeks of IPL 2026 with an injury, confirming the official IPL 2026 schedule with the Indian Express as the source of this information.

## Key Developments

### ✅ MS Dhoni
- **Status:** Not playing IPL 2026 due to injury
- **Reason:** Confirmed injury in first two weeks
- **Source:** The Indian Express

### ❌ IPL 2026 Schedule
- **Team:** Chennai Super Kings (CSK)
- **Date:** 2026
- **Status:** No new schedule details available

## Current Status

**Player**
- MS Dhoni (not playing IPL 2026 due to injury)

**Team**
- Chennai Super Kings (CSK)
- Status: Playing IPL 2026 schedule

## Key Takeaway

The news confirms MS Dhoni's injury status for the upcoming IPL 2026 season, which has already established the schedule.

For more details or to check further about the team's official schedule, please note that specific player news and schedule details are not publicly available thro

## Step 7: Multiple Tools: Let the LLM Choose

Real applications rarely have just one tool. You register multiple tools and the LLM automatically picks the right one based on the user's question.

This is powerful because the user does not need to say "use the weather tool", they just ask a natural question and the LLM figures out which tool to call.

### Tool 2: Real-Time Weather

We add a weather tool using the Open-Meteo API (free, no API key needed).

In [60]:
# --- Tool 2: Get current weather ---
def get_weather(city: str) -> str:
    """
    Tool: Fetches the current weather for a city.
    Uses the Open-Meteo free API — no API key required.

    Parameters:
        city (str): The name of the city, e.g. 'Pune', 'Mumbai'

    Returns:
        str: Temperature and wind speed for that city right now
    """
    # Step 1: Convert city name to coordinates using the geocoding API
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
    geo_response = requests.get(geo_url).json()

    results = geo_response.get("results", [])
    if not results:
        return f"Could not find the city: {city}. Please check the city name and try again."

    lat  = results[0]["latitude"]
    lon  = results[0]["longitude"]
    name = results[0]["name"]

    # Step 2: Use coordinates to get the actual weather
    weather_url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        f"&current_weather=true"
    )
    weather_response = requests.get(weather_url).json()
    current          = weather_response.get("current_weather", {})

    temp      = current.get("temperature", "N/A")
    windspeed = current.get("windspeed", "N/A")

    return f"Current weather in {name}: {temp}°C, Wind speed: {windspeed} km/h"


# Test the tool directly before hooking it into the LLM


In [62]:
print(get_weather("Jamshedpur"))

Current weather in Jamshedpur: 30.6°C, Wind speed: 4.0 km/h


### Register Both Tools

We now describe both tools and build the tools map (connecting tool names to Python functions).

In [63]:
# --- Describe both tools for the LLM ---
multi_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_news",
            "description": "Fetches the latest news headline for a given topic from the internet. Use this when the user asks about recent events, current news, or anything that requires up-to-date information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The topic to search news for. Examples: 'IPL Cricket', 'AI developments', 'Indian economy'"
                    }
                },
                "required": ["topic"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Fetches the current real-time weather for a given city. Use this when the user asks about weather, temperature, or climate conditions right now.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name to get weather for. Examples: 'Pune', 'Mumbai', 'Delhi', 'Bangalore'"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# --- Map tool names to actual Python functions ---
multi_tools_map = {
    "get_news":    get_news,
    "get_weather": get_weather
}

print("Two tools registered and ready.")

Two tools registered and ready.


### Reusable Helper Function

Instead of writing the full tool-calling flow every time, we wrap it in a reusable function. This is good software design — write once, use many times.

In [69]:
def chat_with_tools(user_question: str, tools: list, tools_map: dict) -> str:
    """
    A reusable function that handles the complete tool-calling flow:
    1. Sends the user question to the LLM with available tool descriptions
    2. Checks if the LLM wants to call a tool
    3. Runs the tool and collects the result
    4. Sends the result back to the LLM
    5. Returns the final natural language answer

    Parameters:
        user_question (str): What the user wants to know
        tools (list): List of tool descriptions in Ollama format
        tools_map (dict): Mapping of tool name -> Python function

    Returns:
        str: The LLM's final answer
    """
    messages = [{"role": "user", "content": user_question}]

    # Round 1: Ask LLM — does it need a tool?
    response = ollama.chat(
        model="qwen3.5:0.8b",
        messages=messages,
        tools=tools
    )

    assistant_msg = response["message"]

    # If LLM requested a tool call
    if assistant_msg.get("tool_calls"):
        messages.append(assistant_msg)

        for tool_call in assistant_msg["tool_calls"]:
            name   = tool_call["function"]["name"]
            args   = tool_call["function"]["arguments"]
            result = tools_map[name](**args)

            print(f"  Tool used  : {name}")
            print(f"  Arguments  : {args}")
            print(f"  Tool result: {result}")
            print()

            messages.append({"role": "tool", "content": result})

        # Round 2: Ask LLM again, now give it the tool result
        final = ollama.chat(model="qwen3.5:0.8b", messages=messages)
        return final["message"]["content"]

    else:
        # LLM answered directly (no tool needed)
        return assistant_msg["content"]


print("chat_with_tools() is ready to use.")

chat_with_tools() is ready to use.


### Test with Different Questions

Notice how the LLM picks the correct tool based on the question — without us telling it which one to use.

In [65]:
# --- Question 1: Weather (should trigger get_weather) ---
question = "What is the weather in Pune today?"
print(f"Question: {question}")

print("-" * 60)

answer = chat_with_tools(question, multi_tools, multi_tools_map)
print(f"Answer: {answer}")

Question: What is the weather in Pune today?
------------------------------------------------------------
  Tool used  : get_weather
  Arguments  : {'city': 'Pune'}
  Tool result: Current weather in Pune: 29.0°C, Wind speed: 4.4 km/h

Answer: Today's weather forecast for Pune is as follows:

**Temperature:** 29.0°C
**Wind Speed:** 4.4 km/h

The current weather appears to be pleasant and mild, suitable for most outdoor activities and travel.


In [66]:
# --- Question 2: News (should trigger get_news) ---
question = "Tell me the latest news about Artificial Intelligence."
print(f"Question: {question}")
print("-" * 60)
answer = chat_with_tools(question, multi_tools, multi_tools_map)
print(f"Answer: {answer}")

Question: Tell me the latest news about Artificial Intelligence.
------------------------------------------------------------
  Tool used  : get_news
  Arguments  : {'topic': 'Artificial Intelligence'}
  Tool result: Top headline on 'Artificial Intelligence': Nvidia CEO Jensen Huang says ‘I think we’ve achieved AGI’ (Source: The Verge)

Answer: Great! The latest news about **Artificial Intelligence** has been highlighted. Specifically, it features a key headline from *The Verge* announcing that Nvidia CEO Jensen Huang claims that "we've achieved AGI" (Artificial General Intelligence). This milestone is considered significant for AI development and is often discussed in the context of the AI revolution.

If you're interested in more details—such as the technical challenges, economic implications, or future impact of this achievement—feel free to ask!


In [68]:
# --- Question 3: No tool needed (LLM should answer from its own knowledge) ---
question = "Explain what a neural network is in simple terms."
print(f"Question: {question}")
print("-" * 60)
answer = chat_with_tools(question, multi_tools, multi_tools_map)
print(f"Answer: {answer}")
print()
print("Notice: No tool was used, the LLM answered from its training data.")

Question: Explain what a neural network is in simple terms.
------------------------------------------------------------
  Tool used  : get_news
  Arguments  : {'topic': 'neural network'}
  Tool result: Top headline on 'neural network': The AI tech my dad helped pioneer is now the foundation for the tools I build at AT&T (Source: Business Insider)

Answer: 

Notice: No tool was used, the LLM answered from its training data.


## Summary: What We Learned Today

### The Core Concepts

| Concept | What It Means |
|---------|--------------|
| LLM | A smart model that reasons over text but has a fixed knowledge cutoff |
| Knowledge Cutoff | The date after which the LLM has no information about the world |
| Tool | A Python function that fetches real-world, real-time data |
| Tool Description | A JSON structure that tells the LLM what the tool does and when to use it |
| Tool Calling Flow | User asks → LLM decides → Tool runs → LLM answers |
| Tools Map | A dictionary connecting tool names (strings) to actual Python functions |

### When to Use Tools vs. Other Approaches

| You Need | Use |
|---------|-----|
| Real-time data (weather, news, prices) | Tools |
| Knowledge from a specific document | RAG (Retrieval Augmented Generation) |
| Memory across many conversations | Vector database + memory layer |
| Multi-step autonomous actions | Agents |
| The model to learn new behaviour | Fine-tuning |

### The Tool Calling Flow

```
+-------------+      +------------------+      +------------------+      +------------------+
|    User     | -->  |    LLM           | -->  |    Your Code     | -->  |    LLM           |
|  Question   |      |  (Decides if     |      |  (Runs the tool, |      |  (Reads result,  |
|             |      |   tool needed)   |      |   calls API)     |      |   gives answer)  |
+-------------+      +------------------+      +------------------+      +------------------+
```

### Next Steps

Try building your own tools! Here are some ideas:

- **Stock prices**: Use Yahoo Finance or Alpha Vantage API
- **Cricket scores**: Use a sports API
- **Currency exchange**: Use a free forex API
- **Wikipedia summaries**: Use the Wikipedia API (free, no key needed)
- **Calculator**: A tool that evaluates math expressions safely
- **Database lookup**: A tool that queries your own SQLite database

The pattern is always the same: write the Python function, describe it in JSON, add it to your tools list and tools map, and the LLM will use it automatically.